# Week 5 Exercise - RAG-Based Technical Documentation Assistant

A Gradio-powered RAG (Retrieval Augmented Generation) system that answers questions about technical documentation.

## Features
- Generate synthetic technical documentation using LLMs
- Chunk documents and create vector embeddings
- Retrieve relevant context for user questions
- Answer questions using RAG pipeline
- Interactive Gradio chat interface

## Use Case
Technical documentation assistant for a fictional cloud platform called "CloudLLM"

In [ ]:
# Imports

import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
# Environment Setup

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"API Key exists and begins {openai_api_key[:8]}")
else:
    print("API Key not set")

# Setup client (works with OpenRouter or direct OpenAI)
if openai_api_key and openai_api_key.startswith('sk-or-'):
    client = OpenAI(
        api_key=openai_api_key,
        base_url="https://openrouter.ai/api/v1"
    )
    MODEL = "gpt-4.1-mini"
    print("Using OpenRouter")
else:
    client = OpenAI()
    MODEL = "gpt-4o-mini"
    print("Using OpenAI directly")

# Configuration
DB_NAME = "cloudllm_vector_db"
KNOWLEDGE_BASE_DIR = "cloudllm_knowledge_base"

In [ ]:
# Create Synthetic Knowledge Base

# CloudLLM - A fictional cloud platform for LLM deployment
KNOWLEDGE_BASE = {
    "products": {
        "cloudllm_inference.md": """
# CloudLLM Inference

CloudLLM Inference is our flagship product for deploying and serving large language models at scale.

## Features
- **Auto-scaling**: Automatically scales from 0 to 1000+ instances based on traffic
- **Multi-model support**: Deploy GPT, Claude, Llama, Mistral, and custom models
- **Low latency**: Average response time under 100ms for most models
- **Cost optimization**: Pay only for what you use with per-token billing

## Pricing
- Starter: $0.001 per 1K tokens (input) / $0.002 per 1K tokens (output)
- Pro: $0.0008 per 1K tokens with volume discounts
- Enterprise: Custom pricing with dedicated infrastructure

## Getting Started
1. Create a CloudLLM account at dashboard.cloudllm.io
2. Generate an API key in Settings > API Keys
3. Install the SDK: `pip install cloudllm`
4. Make your first request:

```python
from cloudllm import Client
client = Client(api_key="your-api-key")
response = client.inference.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
```
""",
        "cloudllm_embeddings.md": """
# CloudLLM Embeddings

CloudLLM Embeddings provides high-quality vector embeddings for semantic search and RAG applications.

## Supported Models
- **text-embedding-3-large**: 3072 dimensions, best quality
- **text-embedding-3-small**: 1536 dimensions, cost-effective
- **all-MiniLM-L6-v2**: 384 dimensions, fastest

## Pricing
- text-embedding-3-large: $0.00013 per 1K tokens
- text-embedding-3-small: $0.00002 per 1K tokens
- all-MiniLM-L6-v2: Free (self-hosted option available)

## Usage Example
```python
from cloudllm import Client
client = Client(api_key="your-api-key")

embeddings = client.embeddings.create(
    model="text-embedding-3-small",
    input=["Hello world", "How are you?"]
)
```

## Best Practices
- Batch requests for better throughput (up to 100 texts per request)
- Use text-embedding-3-small for most use cases
- Cache embeddings to reduce costs
""",
        "cloudllm_vectordb.md": """
# CloudLLM VectorDB

CloudLLM VectorDB is a fully managed vector database for storing and querying embeddings.

## Features
- **Managed service**: No infrastructure to manage
- **High performance**: Sub-10ms query latency
- **Scalable**: Supports billions of vectors
- **Metadata filtering**: Filter results by metadata fields

## Pricing
- Starter: Free up to 100K vectors
- Pro: $0.25 per 1M vectors/month
- Enterprise: Custom pricing

## Operations
```python
from cloudllm import Client
client = Client(api_key="your-api-key")

# Create collection
collection = client.vectordb.create_collection("my-docs")

# Upsert vectors
collection.upsert(
    ids=["doc1", "doc2"],
    embeddings=[[0.1, 0.2, ...], [0.3, 0.4, ...]],
    documents=["Hello world", "How are you?"],
    metadatas=[{"source": "web"}, {"source": "pdf"}]
)

# Query
results = collection.query(
    query_embeddings=[[0.1, 0.2, ...]],
    n_results=5
)
```
"""
    },
    "guides": {
        "quickstart.md": """
# CloudLLM Quickstart Guide

Get started with CloudLLM in 5 minutes.

## Prerequisites
- Python 3.8+
- CloudLLM account (sign up at cloudllm.io)
- API key

## Installation
```bash
pip install cloudllm
```

## Authentication
Set your API key as an environment variable:
```bash
export CLOUDLLM_API_KEY="your-api-key"
```

Or pass it directly:
```python
from cloudllm import Client
client = Client(api_key="your-api-key")
```

## Your First Request
```python
response = client.inference.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response.choices[0].message.content)
```

## Next Steps
- Explore the [API Reference](/docs/api)
- Try [CloudLLM Embeddings](/docs/embeddings)
- Build a [RAG Application](/docs/rag-tutorial)
""",
        "rag_tutorial.md": """
# Building RAG Applications with CloudLLM

Learn how to build a Retrieval Augmented Generation (RAG) application.

## What is RAG?
RAG combines document retrieval with LLM generation to provide accurate, contextual answers.

## Architecture
1. **Indexing**: Chunk documents, create embeddings, store in vector DB
2. **Retrieval**: Find relevant chunks for a user query
3. **Generation**: Use retrieved context to generate an answer

## Step 1: Index Documents
```python
from cloudllm import Client

client = Client()
collection = client.vectordb.create_collection("my-docs")

# Chunk and embed documents
for doc in documents:
    chunks = chunk_text(doc, chunk_size=500)
    embeddings = client.embeddings.create(model="text-embedding-3-small", input=chunks)
    collection.upsert(ids=chunk_ids, embeddings=embeddings, documents=chunks)
```

## Step 2: Query and Generate
```python
def answer_question(question):
    # Get query embedding
    query_emb = client.embeddings.create(model="text-embedding-3-small", input=[question])
    
    # Retrieve relevant chunks
    results = collection.query(query_embeddings=query_emb, n_results=5)
    context = "\n\n".join(results.documents[0])
    
    # Generate answer
    response = client.inference.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"Answer based on this context:\n{context}"},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content
```

## Best Practices
- Use chunk sizes of 500-1000 characters with 100-200 overlap
- Retrieve 3-5 relevant chunks per query
- Include source citations in responses
- Consider query rewriting for complex questions
"""
    },
    "company": {
        "about.md": """
# About CloudLLM

CloudLLM is a leading cloud platform for deploying and scaling large language models.

## Our Mission
To make AI accessible to every developer and organization.

## Founded
2023 in San Francisco, CA

## Team
- **Sarah Chen** - CEO & Co-founder (ex-Google AI)
- **Marcus Johnson** - CTO & Co-founder (ex-OpenAI)
- **Priya Patel** - VP Engineering (ex-AWS)
- 50+ engineers and researchers

## Customers
- 10,000+ developers
- 500+ enterprise customers
- Fortune 500 companies in finance, healthcare, and tech

## Contact
- Email: support@cloudllm.io
- Twitter: @cloudllm
- GitHub: github.com/cloudllm
""",
        "support.md": """
# CloudLLM Support

## Getting Help

### Documentation
Visit docs.cloudllm.io for comprehensive guides and API reference.

### Community
- Discord: discord.gg/cloudllm
- GitHub Discussions: github.com/cloudllm/discussions

### Email Support
- Free tier: community@cloudllm.io (48-hour response)
- Pro tier: support@cloudllm.io (24-hour response)
- Enterprise: Dedicated support engineer, 1-hour response SLA

## Common Issues

### Rate Limiting
Default limits: 1000 requests/minute for Pro, 100 for Starter.
Contact sales for higher limits.

### API Key Issues
- Regenerate keys at dashboard.cloudllm.io/settings/api-keys
- Keys starting with `cllm_` are valid
- Never share keys in public repositories

### Billing
View usage at dashboard.cloudllm.io/billing
Set spending alerts in Settings > Billing > Alerts
"""
    }
}

# Create knowledge base directory and files
def create_knowledge_base():
    """Create the knowledge base files."""
    base_path = Path(KNOWLEDGE_BASE_DIR)
    
    for category, files in KNOWLEDGE_BASE.items():
        category_path = base_path / category
        category_path.mkdir(parents=True, exist_ok=True)
        
        for filename, content in files.items():
            file_path = category_path / filename
            with open(file_path, "w") as f:
                f.write(content.strip())
    
    print(f"Knowledge base created at {KNOWLEDGE_BASE_DIR}/")
    
create_knowledge_base()

In [ ]:
# Load and Chunk Documents

def load_documents():
    """Load all markdown files from knowledge base."""
    documents = []
    base_path = Path(KNOWLEDGE_BASE_DIR)
    
    for md_file in base_path.rglob("*.md"):
        with open(md_file, "r") as f:
            content = f.read()
        
        # Create document with metadata
        doc = Document(
            page_content=content,
            metadata={
                "source": str(md_file),
                "category": md_file.parent.name,
                "filename": md_file.name
            }
        )
        documents.append(doc)
    
    return documents

# Load documents
documents = load_documents()
print(f"Loaded {len(documents)} documents")

# Chunk documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n## ", "\n### ", "\n\n", "\n", " "]
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

# Preview a chunk
print(f"\nSample chunk:")
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

In [ ]:
# Create Vector Store

# Use HuggingFace embeddings (free, runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Create Chroma vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

print(f"Vector store created with {len(chunks)} vectors")
print(f"Persisted to {DB_NAME}/")

In [ ]:
# Create Retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Return top 4 most relevant chunks
)

# Test retrieval
test_query = "How do I get started with CloudLLM?"
results = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(results)} chunks:")
for i, doc in enumerate(results, 1):
    print(f"\n--- Chunk {i} ({doc.metadata['filename']}) ---")
    print(doc.page_content[:200] + "...")

In [ ]:
# RAG Pipeline

SYSTEM_PROMPT = """
You are a helpful technical support assistant for CloudLLM, a cloud platform for deploying LLMs.

Guidelines:
- Answer questions based on the provided context
- Be concise but thorough
- Include code examples when relevant
- If the context doesn't contain enough information, say so
- Suggest relevant documentation pages when appropriate

Context from CloudLLM documentation:
{context}
"""

def answer_question(question: str, history: list) -> str:
    """Answer a question using RAG pipeline."""
    
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    
    # Build context from retrieved documents
    context_parts = []
    for doc in docs:
        source = doc.metadata.get('filename', 'unknown')
        context_parts.append(f"[Source: {source}]\n{doc.page_content}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Create messages
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
        {"role": "user", "content": question}
    ]
    
    # Generate response
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=0.3,
            max_tokens=1000
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"


# Test the RAG pipeline
test_question = "What is CloudLLM Inference and how much does it cost?"
answer = answer_question(test_question, [])
print(f"Q: {test_question}")
print(f"\nA: {answer}")

In [ ]:
# Gradio UI

EXAMPLE_QUESTIONS = [
    "How do I get started with CloudLLM?",
    "What embedding models are available?",
    "How much does CloudLLM Inference cost?",
    "How do I build a RAG application?",
    "Who founded CloudLLM?",
    "How do I contact support?",
    "What is the rate limit for the API?"
]

with gr.Blocks(title="CloudLLM Documentation Assistant", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # CloudLLM Documentation Assistant
    
    Ask questions about CloudLLM products, pricing, and usage.
    
    This assistant uses RAG (Retrieval Augmented Generation) to find relevant 
    documentation and provide accurate answers.
    """)
    
    chatbot = gr.ChatInterface(
        fn=answer_question,
        type="messages",
        examples=EXAMPLE_QUESTIONS,
        title=None
    )
    
    gr.Markdown("""
    ### About This Demo
    - **Knowledge Base**: 7 markdown documents about CloudLLM (fictional)
    - **Embeddings**: HuggingFace all-MiniLM-L6-v2 (384 dimensions)
    - **Vector Store**: Chroma (local)
    - **LLM**: GPT-4.1-mini via OpenRouter
    """)

In [ ]:
# Launch the app
app.launch()

## Technical Notes

### RAG Pipeline Overview
1. **Document Loading**: Read markdown files from knowledge base
2. **Chunking**: Split into 500-char chunks with 100-char overlap
3. **Embedding**: Convert chunks to vectors using all-MiniLM-L6-v2
4. **Storage**: Store in Chroma vector database
5. **Retrieval**: Find top-4 similar chunks for each query
6. **Generation**: Use LLM to answer based on retrieved context

### Key Concepts from Week 5
- **Chunking Strategy**: RecursiveCharacterTextSplitter respects document structure
- **Overlap**: Ensures context isn't lost at chunk boundaries
- **Embedding Model**: HuggingFace models are free and run locally
- **Vector Store**: Chroma persists embeddings to disk
- **Retriever**: LangChain abstraction for similarity search
- **System Prompt**: Instructs LLM how to use the context

### Extending This Example
- Add more documents to the knowledge base
- Implement query rewriting for better retrieval
- Add metadata filtering (by category, date, etc.)
- Implement reranking for improved relevance
- Add evaluation metrics (MRR, NDCG)